# Data Preprocessing & Pipeline Preparation

This notebook cleans the text data for natural language processing and standardizes the network graph data in preparation for GraphFrames analysis.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lower, regexp_replace, trim, when

# Initialize Spark Session
spark = SparkSession.builder.appName("Preprocessing_MentalHealth").getOrCreate()

# Load datasets
mental_health_df = spark.read.csv("mental_health.csv", header=True, inferSchema=True)
twitter_graph_df = spark.read.csv("twitter_graph.csv", header=True, inferSchema=True)

### Text Cleaning
Standardizing the text fields by converting to lowercase, removing non-alphanumeric characters, and stripping excess whitespace. This ensures consistency when pushing data into Spark MLlib pipelines.

In [ ]:
cleaned_text_df = mental_health_df.withColumn("Cleaned_Text", lower(col("Text")))
cleaned_text_df = cleaned_text_df.withColumn("Cleaned_Text", regexp_replace(col("Cleaned_Text"), "[^a-zA-Z\\s]", ""))
cleaned_text_df = cleaned_text_df.withColumn("Cleaned_Text", trim(col("Cleaned_Text")))

cleaned_text_df.select("Text", "Cleaned_Text", "Emotion").show(5, truncate=False)

### Graph Data Formatting
Filtering out incomplete edge relationships and enforcing appropriate data types for network metrics, ensuring compatibility with GraphFrames.

In [ ]:
clean_graph_df = twitter_graph_df.dropna(subset=["Source", "Destination"])

# Standardize boolean flags for verification status
clean_graph_df = clean_graph_df.withColumn("Source Verified", when(col("Source Verified") == "TRUE", True).otherwise(False))
clean_graph_df = clean_graph_df.withColumn("Destination Verified", when(col("Destination Verified") == "TRUE", True).otherwise(False))

clean_graph_df.printSchema()

### Data Export
Saving the cleaned datasets to Parquet format to optimize read performance for downstream modeling tasks in Databricks.

In [ ]:
# Uncomment to execute write operations in the Databricks environment
# cleaned_text_df.write.parquet("dbfs:/FileStore/cleaned_mental_health.parquet", mode="overwrite")
# clean_graph_df.write.parquet("dbfs:/FileStore/cleaned_twitter_graph.parquet", mode="overwrite")
print("Preprocessing pipelines defined. Ready for export.")